In [7]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "GPU" if torch.cuda.is_available() else "CPU (Mac)")

PyTorch version: 2.12.0
CUDA available: False
Device: CPU (Mac)


In [8]:
train_df = pd.read_csv('../data/train_clean.csv').dropna(subset=['clean_text'])
test_df  = pd.read_csv('../data/test_clean.csv').dropna(subset=['clean_text'])

# Use 2000 train / 500 test for speed on CPU
train_small = train_df.sample(2000, random_state=42).reset_index(drop=True)
test_small  = test_df.sample(500, random_state=42).reset_index(drop=True)

print(f"Train subset: {len(train_small)}")
print(f"Test subset:  {len(test_small)}")
print(f"Label balance:\n{train_small['label'].value_counts()}")

Train subset: 2000
Test subset:  500
Label balance:
label
0    1040
1     960
Name: count, dtype: int64


In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("Tokenizer loaded")

# Test it
sample = "This movie was absolutely fantastic"
tokens = tokenizer(sample, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
print("Input IDs shape:", tokens['input_ids'].shape)

Tokenizer loaded
Input IDs shape: torch.Size([1, 128])


In [10]:
class IMDbDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IMDbDataset(train_small['clean_text'].tolist(), 
                             train_small['label'].tolist(), tokenizer)
test_dataset  = IMDbDataset(test_small['clean_text'].tolist(),  
                             test_small['label'].tolist(),  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 125
Test batches:  32


In [11]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)
device = torch.device('cpu')
model = model.to(device)
print("BERT model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded!
Parameters: 109,483,778


In [12]:
optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 2

train_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, 
                        attention_mask=attention_mask, 
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"\nEpoch {epoch+1} complete | Avg Loss: {avg_loss:.4f}\n")

print("Training complete")

Epoch 1 | Batch 0/125 | Loss: 0.7504
Epoch 1 | Batch 20/125 | Loss: 0.7839
Epoch 1 | Batch 40/125 | Loss: 0.6897
Epoch 1 | Batch 60/125 | Loss: 0.4816
Epoch 1 | Batch 80/125 | Loss: 0.5867
Epoch 1 | Batch 100/125 | Loss: 0.4080
Epoch 1 | Batch 120/125 | Loss: 0.3140

Epoch 1 complete | Avg Loss: 0.5576

Epoch 2 | Batch 0/125 | Loss: 0.4674
Epoch 2 | Batch 20/125 | Loss: 0.3152
Epoch 2 | Batch 40/125 | Loss: 0.1163
Epoch 2 | Batch 60/125 | Loss: 0.6814
Epoch 2 | Batch 80/125 | Loss: 0.1629
Epoch 2 | Batch 100/125 | Loss: 0.1499
Epoch 2 | Batch 120/125 | Loss: 0.3889

Epoch 2 complete | Avg Loss: 0.2939

Training complete


In [13]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("BERT Accuracy:", round(accuracy_score(all_labels, all_preds) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Negative', 'Positive']))

BERT Accuracy: 85.0 %

Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.82      0.85       266
    Positive       0.81      0.88      0.85       234

    accuracy                           0.85       500
   macro avg       0.85      0.85      0.85       500
weighted avg       0.85      0.85      0.85       500



In [14]:
model.save_pretrained('../models/bert_sentiment')
tokenizer.save_pretrained('../models/bert_sentiment')
print("BERT model saved to models/bert_sentiment/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT model saved to models/bert_sentiment/
